# CNNs from Scratch

**Goal:** Implement Conv2D (naive loops + im2col), pooling layers, and a small trainable CNN using only NumPy.

## Key Formulas

**Output spatial size:**
$$H_{out} = \frac{H_{in} - K + 2P}{S} + 1$$

where $K$ = kernel size, $P$ = padding, $S$ = stride.

**Receptive field** at layer $l$: $r_l = r_{l-1} + (K_l - 1) \cdot \prod_{i=1}^{l-1} S_i$

**Number of parameters** in a conv layer: $C_{out} \times (C_{in} \times K \times K + 1)$ (with bias).

**Why convolutions over FC?**
- **Parameter sharing**: same filter applied everywhere = far fewer parameters
- **Translation equivariance**: feature detected regardless of position
- **Local connectivity**: exploits spatial structure

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Conv2D Forward Pass (Naive Loops)

Input shape: $(N, C_{in}, H, W)$, Filter shape: $(C_{out}, C_{in}, K_H, K_W)$

$$y[n, f, i, j] = \sum_{c} \sum_{p} \sum_{q} x[n, c, i \cdot S + p, j \cdot S + q] \cdot w[f, c, p, q] + b[f]$$

In [ ]:
def conv2d_forward_naive(x, w, b, stride=1, pad=0):
    """Conv2D forward pass using explicit loops.
    
    Args:
        x: input (N, C_in, H, W)
        w: filters (C_out, C_in, KH, KW)
        b: bias (C_out,)
        stride: int
        pad: int (zero-padding)
    Returns:
        out: (N, C_out, H_out, W_out)
    """
    N, C_in, H, W = x.shape
    C_out, _, KH, KW = w.shape
    
    # Pad input
    if pad > 0:
        x_pad = np.pad(x, ((0,0), (0,0), (pad,pad), (pad,pad)), mode='constant')
    else:
        x_pad = x
    
    H_out = (H + 2*pad - KH) // stride + 1
    W_out = (W + 2*pad - KW) // stride + 1
    out = np.zeros((N, C_out, H_out, W_out))
    
    for n in range(N):             # batch
        for f in range(C_out):     # filter
            for i in range(H_out): # height
                for j in range(W_out): # width
                    h_start = i * stride
                    w_start = j * stride
                    receptive_field = x_pad[n, :, h_start:h_start+KH, w_start:w_start+KW]
                    out[n, f, i, j] = np.sum(receptive_field * w[f]) + b[f]
    
    cache = (x, x_pad, w, b, stride, pad)
    return out, cache

# Quick test
x_test = np.random.randn(2, 3, 8, 8)  # batch=2, channels=3, 8x8
w_test = np.random.randn(4, 3, 3, 3)  # 4 filters, 3x3
b_test = np.zeros(4)

out, cache = conv2d_forward_naive(x_test, w_test, b_test, stride=1, pad=1)
print(f"Input:  {x_test.shape}")
print(f"Output: {out.shape}  (expected (2, 4, 8, 8) with pad=1, stride=1)")

## 2. Conv2D Forward Pass (im2col)

The key optimization: reshape input patches into columns, then convolution = matrix multiplication.

1. Extract all receptive field patches -> column matrix of shape $(C_{in} \cdot K_H \cdot K_W, H_{out} \cdot W_{out})$
2. Reshape filters -> $(C_{out}, C_{in} \cdot K_H \cdot K_W)$
3. Output = filters @ cols (a single GEMM!)

In [ ]:
def im2col(x_pad, KH, KW, stride, H_out, W_out):
    """Convert image patches to columns.
    
    Args:
        x_pad: (C_in, H_padded, W_padded) single image
        KH, KW: kernel size
    Returns:
        cols: (C_in * KH * KW, H_out * W_out)
    """
    C_in = x_pad.shape[0]
    cols = np.zeros((C_in * KH * KW, H_out * W_out))
    col_idx = 0
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * stride
            w_start = j * stride
            patch = x_pad[:, h_start:h_start+KH, w_start:w_start+KW]  # (C_in, KH, KW)
            cols[:, col_idx] = patch.reshape(-1)
            col_idx += 1
    return cols

def col2im(cols, C_in, H, W, KH, KW, stride, pad, H_out, W_out):
    """Inverse of im2col — scatter column gradients back to image."""
    H_pad, W_pad = H + 2*pad, W + 2*pad
    x_pad = np.zeros((C_in, H_pad, W_pad))
    col_idx = 0
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * stride
            w_start = j * stride
            patch = cols[:, col_idx].reshape(C_in, KH, KW)
            x_pad[:, h_start:h_start+KH, w_start:w_start+KW] += patch
            col_idx += 1
    if pad > 0:
        return x_pad[:, pad:-pad, pad:-pad]
    return x_pad

def conv2d_forward_im2col(x, w, b, stride=1, pad=0):
    """Conv2D forward using im2col + GEMM."""
    N, C_in, H, W = x.shape
    C_out, _, KH, KW = w.shape
    H_out = (H + 2*pad - KH) // stride + 1
    W_out = (W + 2*pad - KW) // stride + 1
    
    if pad > 0:
        x_pad = np.pad(x, ((0,0),(0,0),(pad,pad),(pad,pad)), mode='constant')
    else:
        x_pad = x
    
    w_row = w.reshape(C_out, -1)  # (C_out, C_in*KH*KW)
    out = np.zeros((N, C_out, H_out, W_out))
    
    col_cache = []
    for n in range(N):
        cols = im2col(x_pad[n], KH, KW, stride, H_out, W_out)
        col_cache.append(cols)
        result = w_row @ cols + b.reshape(-1, 1)  # (C_out, H_out*W_out)
        out[n] = result.reshape(C_out, H_out, W_out)
    
    cache = (x, x_pad, w, b, stride, pad, col_cache, H_out, W_out)
    return out, cache

# Verify both methods give same result
out_im2col, cache2 = conv2d_forward_im2col(x_test, w_test, b_test, stride=1, pad=1)
print(f"Max difference naive vs im2col: {np.max(np.abs(out - out_im2col)):.2e}")

## 3. Conv2D Backward Pass

We need three gradients:
- $\frac{\partial L}{\partial w}$: gradient w.r.t. filters
- $\frac{\partial L}{\partial b}$: gradient w.r.t. biases
- $\frac{\partial L}{\partial x}$: gradient w.r.t. input (for backprop to previous layer)

Using im2col representation:
- $\text{out} = W_{row} \cdot \text{cols} + b$ where $W_{row}$ is reshaped filters
- $dW_{row} = d\text{out} \cdot \text{cols}^T$
- $d\text{cols} = W_{row}^T \cdot d\text{out}$ then `col2im` to get $dx$
- $db = \sum d\text{out}$

In [ ]:
def conv2d_backward(dout, cache):
    """Conv2D backward pass using im2col.
    
    Args:
        dout: upstream gradient (N, C_out, H_out, W_out)
        cache: from forward pass
    Returns:
        dx, dw, db
    """
    x, x_pad, w, b, stride, pad, col_cache, H_out, W_out = cache
    N, C_in, H, W = x.shape
    C_out, _, KH, KW = w.shape
    
    w_row = w.reshape(C_out, -1)
    dw_row = np.zeros_like(w_row)
    db = np.zeros_like(b)
    dx = np.zeros_like(x)
    
    for n in range(N):
        dout_reshaped = dout[n].reshape(C_out, -1)  # (C_out, H_out*W_out)
        cols = col_cache[n]
        
        # Gradient w.r.t. weights: dW = dout @ cols^T
        dw_row += dout_reshaped @ cols.T
        
        # Gradient w.r.t. bias: db = sum over spatial dims
        db += dout_reshaped.sum(axis=1)
        
        # Gradient w.r.t. input: dcols = W^T @ dout, then col2im
        dcols = w_row.T @ dout_reshaped  # (C_in*KH*KW, H_out*W_out)
        dx[n] = col2im(dcols, C_in, H, W, KH, KW, stride, pad, H_out, W_out)
    
    dw = dw_row.reshape(w.shape)
    return dx, dw, db

# Gradient check
dout_test = np.random.randn(*out_im2col.shape)
dx, dw, db = conv2d_backward(dout_test, cache2)
print(f"dx shape: {dx.shape} (should be {x_test.shape})")
print(f"dw shape: {dw.shape} (should be {w_test.shape})")
print(f"db shape: {db.shape} (should be {b_test.shape})")

# Numerical gradient check for dw
eps = 1e-5
# Check one element of w
idx = (0, 0, 1, 1)
w_plus = w_test.copy(); w_plus[idx] += eps
w_minus = w_test.copy(); w_minus[idx] -= eps
out_plus, _ = conv2d_forward_im2col(x_test, w_plus, b_test, stride=1, pad=1)
out_minus, _ = conv2d_forward_im2col(x_test, w_minus, b_test, stride=1, pad=1)
numerical_grad = np.sum((out_plus - out_minus) * dout_test) / (2 * eps)
print(f"\nGradient check dw{idx}:")
print(f"  Analytical: {dw[idx]:.6f}")
print(f"  Numerical:  {numerical_grad:.6f}")
print(f"  Rel error:  {abs(dw[idx] - numerical_grad) / (abs(numerical_grad) + 1e-8):.2e}")

## 4. MaxPool2D and AvgPool2D

In [ ]:
def maxpool2d_forward(x, pool_size=2, stride=2):
    """Max pooling forward pass.
    
    Args:
        x: (N, C, H, W)
    Returns:
        out: (N, C, H_out, W_out)
    """
    N, C, H, W = x.shape
    H_out = (H - pool_size) // stride + 1
    W_out = (W - pool_size) // stride + 1
    out = np.zeros((N, C, H_out, W_out))
    mask = np.zeros_like(x)  # store argmax positions for backward
    
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * stride
            w_start = j * stride
            window = x[:, :, h_start:h_start+pool_size, w_start:w_start+pool_size]
            out[:, :, i, j] = window.max(axis=(2, 3))
            # Record which element was the max
            window_reshaped = window.reshape(N, C, -1)
            max_idx = window_reshaped.argmax(axis=2)
            max_r, max_c = np.unravel_index(max_idx, (pool_size, pool_size))
            for n in range(N):
                for c in range(C):
                    mask[n, c, h_start + max_r[n,c], w_start + max_c[n,c]] = 1
    
    cache = (x, mask, pool_size, stride)
    return out, cache

def maxpool2d_backward(dout, cache):
    """Max pooling backward: gradient flows only through the max element."""
    x, mask, pool_size, stride = cache
    N, C, H, W = x.shape
    H_out, W_out = dout.shape[2], dout.shape[3]
    dx = np.zeros_like(x)
    
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * stride
            w_start = j * stride
            dx[:, :, h_start:h_start+pool_size, w_start:w_start+pool_size] += (
                mask[:, :, h_start:h_start+pool_size, w_start:w_start+pool_size] *
                dout[:, :, i:i+1, j:j+1]
            )
    return dx

def avgpool2d_forward(x, pool_size=2, stride=2):
    """Average pooling forward pass."""
    N, C, H, W = x.shape
    H_out = (H - pool_size) // stride + 1
    W_out = (W - pool_size) // stride + 1
    out = np.zeros((N, C, H_out, W_out))
    
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * stride
            w_start = j * stride
            out[:, :, i, j] = x[:, :, h_start:h_start+pool_size, 
                                w_start:w_start+pool_size].mean(axis=(2,3))
    
    cache = (x.shape, pool_size, stride)
    return out, cache

def avgpool2d_backward(dout, cache):
    """Average pooling backward: gradient distributed equally."""
    x_shape, pool_size, stride = cache
    N, C, H, W = x_shape
    H_out, W_out = dout.shape[2], dout.shape[3]
    dx = np.zeros(x_shape)
    
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * stride
            w_start = j * stride
            dx[:, :, h_start:h_start+pool_size, w_start:w_start+pool_size] += (
                dout[:, :, i:i+1, j:j+1] / (pool_size * pool_size)
            )
    return dx

# Test pooling
x_pool = np.random.randn(1, 1, 4, 4)
print("Input:")
print(x_pool[0, 0])

out_max, cache_max = maxpool2d_forward(x_pool, pool_size=2, stride=2)
out_avg, cache_avg = avgpool2d_forward(x_pool, pool_size=2, stride=2)
print(f"\nMaxPool output: {out_max[0,0]}")
print(f"AvgPool output: {out_avg[0,0]}")

## 5. ReLU activation

In [ ]:
def relu_forward(x):
    out = np.maximum(0, x)
    cache = x
    return out, cache

def relu_backward(dout, cache):
    x = cache
    dx = dout * (x > 0).astype(float)
    return dx

## 6. Build a Small CNN and Train on Synthetic Data

Architecture: `Conv(1->4, 3x3, pad=1) -> ReLU -> MaxPool(2) -> Conv(4->8, 3x3, pad=1) -> ReLU -> MaxPool(2) -> FC -> Softmax`

Task: classify 16x16 synthetic images into 3 classes (vertical stripes, horizontal stripes, random noise).

In [ ]:
def make_synthetic_data(n_samples=300, size=16):
    """Create synthetic image classification data."""
    X = []
    y = []
    n_per_class = n_samples // 3
    
    for _ in range(n_per_class):
        # Class 0: vertical stripes
        img = np.zeros((1, size, size))
        freq = np.random.randint(2, 5)
        for j in range(size):
            if (j // freq) % 2 == 0:
                img[0, :, j] = 1.0
        img += np.random.randn(1, size, size) * 0.1
        X.append(img); y.append(0)
    
    for _ in range(n_per_class):
        # Class 1: horizontal stripes
        img = np.zeros((1, size, size))
        freq = np.random.randint(2, 5)
        for i in range(size):
            if (i // freq) % 2 == 0:
                img[0, i, :] = 1.0
        img += np.random.randn(1, size, size) * 0.1
        X.append(img); y.append(1)
    
    for _ in range(n_per_class):
        # Class 2: checker pattern
        img = np.zeros((1, size, size))
        freq = np.random.randint(2, 5)
        for i in range(size):
            for j in range(size):
                if ((i // freq) + (j // freq)) % 2 == 0:
                    img[0, i, j] = 1.0
        img += np.random.randn(1, size, size) * 0.1
        X.append(img); y.append(2)
    
    X = np.array(X)
    y = np.array(y)
    # Shuffle
    perm = np.random.permutation(len(y))
    return X[perm], y[perm]

X_train, y_train = make_synthetic_data(300, 16)
X_test, y_test = make_synthetic_data(60, 16)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Class distribution: {np.bincount(y_train)}")

# Show samples
fig, axes = plt.subplots(1, 6, figsize=(12, 2))
labels = ['V-stripes', 'H-stripes', 'Checker']
for i in range(6):
    axes[i].imshow(X_train[i * 50, 0], cmap='gray')
    axes[i].set_title(labels[y_train[i * 50]])
    axes[i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def cross_entropy_loss(probs, y):
    N = len(y)
    log_probs = -np.log(probs[np.arange(N), y] + 1e-12)
    return log_probs.mean()

class SimpleCNN:
    """Conv(1->4) -> ReLU -> Pool -> Conv(4->8) -> ReLU -> Pool -> FC -> Softmax."""
    
    def __init__(self, n_classes=3):
        # He initialization
        self.w1 = np.random.randn(4, 1, 3, 3) * np.sqrt(2.0 / 9)
        self.b1 = np.zeros(4)
        self.w2 = np.random.randn(8, 4, 3, 3) * np.sqrt(2.0 / 36)
        self.b2 = np.zeros(8)
        # After two 2x2 pools on 16x16 -> 4x4, with 8 channels -> 128
        self.w3 = np.random.randn(128, n_classes) * np.sqrt(2.0 / 128)
        self.b3 = np.zeros(n_classes)
    
    def forward(self, x):
        # Conv1 -> ReLU -> Pool
        z1, self.cache_conv1 = conv2d_forward_im2col(x, self.w1, self.b1, stride=1, pad=1)
        a1, self.cache_relu1 = relu_forward(z1)
        p1, self.cache_pool1 = maxpool2d_forward(a1, pool_size=2, stride=2)
        
        # Conv2 -> ReLU -> Pool
        z2, self.cache_conv2 = conv2d_forward_im2col(p1, self.w2, self.b2, stride=1, pad=1)
        a2, self.cache_relu2 = relu_forward(z2)
        p2, self.cache_pool2 = maxpool2d_forward(a2, pool_size=2, stride=2)
        
        # Flatten -> FC
        self.flat_shape = p2.shape
        flat = p2.reshape(p2.shape[0], -1)  # (N, 128)
        logits = flat @ self.w3 + self.b3   # (N, n_classes)
        probs = softmax(logits)
        
        self.flat = flat
        self.probs = probs
        return probs
    
    def backward(self, y, lr=0.001):
        N = len(y)
        # Softmax + CE gradient
        dlogits = self.probs.copy()
        dlogits[np.arange(N), y] -= 1
        dlogits /= N
        
        # FC backward
        dw3 = self.flat.T @ dlogits
        db3 = dlogits.sum(axis=0)
        dflat = dlogits @ self.w3.T
        
        # Reshape back
        dp2 = dflat.reshape(self.flat_shape)
        
        # Pool2 -> ReLU2 -> Conv2
        da2 = maxpool2d_backward(dp2, self.cache_pool2)
        dz2 = relu_backward(da2, self.cache_relu2)
        dp1, dw2, db2 = conv2d_backward(dz2, self.cache_conv2)
        
        # Pool1 -> ReLU1 -> Conv1
        da1 = maxpool2d_backward(dp1, self.cache_pool1)
        dz1 = relu_backward(da1, self.cache_relu1)
        dx, dw1, db1 = conv2d_backward(dz1, self.cache_conv1)
        
        # SGD update
        self.w3 -= lr * dw3; self.b3 -= lr * db3
        self.w2 -= lr * dw2; self.b2 -= lr * db2
        self.w1 -= lr * dw1; self.b1 -= lr * db1

print("SimpleCNN initialized.")

In [ ]:
# Training loop
cnn = SimpleCNN(n_classes=3)
losses = []
accs = []
batch_size = 16
n_epochs = 20
lr = 0.005

for epoch in range(n_epochs):
    perm = np.random.permutation(len(y_train))
    epoch_loss = 0
    n_batches = 0
    for start in range(0, len(y_train), batch_size):
        idx = perm[start:start+batch_size]
        xb, yb = X_train[idx], y_train[idx]
        probs = cnn.forward(xb)
        loss = cross_entropy_loss(probs, yb)
        cnn.backward(yb, lr=lr)
        epoch_loss += loss
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    
    # Accuracy on test set
    test_probs = cnn.forward(X_test)
    test_preds = test_probs.argmax(axis=1)
    acc = (test_preds == y_test).mean()
    accs.append(acc)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}: loss={avg_loss:.4f}, test_acc={acc:.1%}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(losses, 'b-'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Training Loss')
ax2.plot(accs, 'r-'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Test Accuracy')
for ax in (ax1, ax2): ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Visualize Learned Filters and Feature Maps

In [ ]:
# Visualize first-layer filters
fig, axes = plt.subplots(1, 4, figsize=(10, 3))
fig.suptitle('Layer 1 Learned Filters (3x3)', fontsize=12)
for i in range(4):
    axes[i].imshow(cnn.w1[i, 0], cmap='coolwarm', interpolation='nearest')
    axes[i].set_title(f'Filter {i}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

# Visualize feature maps for one sample
sample_idx = 0
x_sample = X_test[sample_idx:sample_idx+1]
z1, _ = conv2d_forward_im2col(x_sample, cnn.w1, cnn.b1, stride=1, pad=1)
a1, _ = relu_forward(z1)

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
axes[0].imshow(x_sample[0, 0], cmap='gray')
axes[0].set_title(f'Input (class={labels[y_test[sample_idx]]})')
axes[0].axis('off')
for i in range(4):
    axes[i+1].imshow(a1[0, i], cmap='viridis')
    axes[i+1].set_title(f'Feature map {i}')
    axes[i+1].axis('off')
fig.suptitle('Input + ReLU Feature Maps after Conv1', fontsize=12)
plt.tight_layout()
plt.show()

## Interview Questions

### Q: "Output size formula?"

$$H_{out} = \left\lfloor\frac{H_{in} + 2P - K}{S}\right\rfloor + 1$$

Example: input 32x32, kernel 5x5, pad=0, stride=1 -> $(32 - 5)/1 + 1 = 28$.

For "same" padding: $P = (K-1)/2$ (only works for odd $K$ with $S=1$).

### Q: "What is the receptive field?"

The region in the **original input** that a single output neuron "sees." It grows with depth:
- Layer 1 (3x3 conv): RF = 3x3
- After 2x2 pool: RF = 4x4 (stride doubles effective RF)
- Layer 2 (3x3 conv after pool): RF = 8x8

Recursive formula: $r_l = r_{l-1} + (K_l - 1) \times \prod_{i=1}^{l-1} S_i$

### Q: "Why convolutions instead of fully-connected layers?"

1. **Parameter efficiency**: A 3x3 conv filter has 9 weights shared across all positions. An FC layer connecting two 32x32 feature maps would need $32^2 \times 32^2 \approx 1M$ parameters.
2. **Translation equivariance**: If the input shifts, the feature map shifts by the same amount. FC layers have no such guarantee.
3. **Exploits spatial locality**: Nearby pixels are more correlated than distant ones.

### Q: "im2col — why is it used?"

It converts convolution into a single large matrix multiply (GEMM), which is highly optimized on CPUs/GPUs. The tradeoff is extra memory for the column matrix (up to $K^2 \times$ expansion).

### Q: "1x1 convolutions — what do they do?"

They act as per-pixel FC layers across channels. Used to change the channel dimension (e.g., bottleneck in ResNet). No spatial mixing, only channel mixing.